# 하림주가 시계열 분석

In [27]:
import yfinance as yf
import pandas as pd
import statsmodels as sm
import matplotlib.pyplot as plt
from pprint import pprint

## 1. 데이터 수집

In [28]:
harim = yf.download('136480.KS', '2016-01-01', '2026-01-01')
kospi = yf.download('^KS11', '2016-01-01', '2026-01-01')

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [29]:
# 날짜를 datetime 타입으로 전환
harim.index = pd.to_datetime(harim.index)

# 연도별 종가 데이터 슬라이싱
harim_total_year = {}
for year in range(2016, 2026):
    harim_total_year[year] = harim['Close'].loc[harim.index.year == year]

print(harim_total_year)

{2016: Ticker        136480.KS
Date                   
2016-01-04  3463.022217
2016-01-05  3416.623291
2016-01-06  3429.277588
2016-01-07  3408.187744
2016-01-08  3387.097412
...                 ...
2016-12-23  4015.587158
2016-12-26  3948.098633
2016-12-27  4057.768066
2016-12-28  4211.212402
2016-12-29  4189.901367

[246 rows x 1 columns], 2017: Ticker        136480.KS
Date                   
2017-01-02  4215.475098
2017-01-03  4241.049805
2017-01-04  4206.950684
2017-01-05  4249.573730
2017-01-06  4262.361328
...                 ...
2017-12-21  2921.117432
2017-12-22  2945.179688
2017-12-26  2978.865967
2017-12-27  2940.367188
2017-12-28  3002.927979

[241 rows x 1 columns], 2018: Ticker        136480.KS
Date                   
2018-01-02  3070.301514
2018-01-03  3051.052002
2018-01-04  2978.865967
2018-01-05  3002.927979
2018-01-08  2959.616699
...                 ...
2018-12-21  2743.059082
2018-12-24  2666.061279
2018-12-26  2579.438477
2018-12-27  2608.312500
2018-12-28  2694.93

In [30]:
# ret(mkt_ret) 추가, 종가(Close)를 기준으로 pct_change(전 행 대비 증가량)
harim['ret'] = harim['Close'].pct_change()
kospi['mkt_ret'] = kospi['Close'].pct_change()
data = harim[['ret']].merge(
    kospi[['mkt_ret']],
    left_index=True,
    right_index=True,
    how='inner'
)
print(data)

Price            ret   mkt_ret
Ticker                        
Date                          
2016-01-04       NaN       NaN
2016-01-05 -0.013398  0.006134
2016-01-06  0.003704 -0.002642
2016-01-07 -0.006150 -0.010959
2016-01-08 -0.006188  0.006979
...              ...       ...
2025-12-23 -0.001650  0.002774
2025-12-24  0.003306 -0.002113
2025-12-26 -0.004942  0.005126
2025-12-29  0.010067  0.022007
2025-12-30  0.001661 -0.001514

[2447 rows x 2 columns]


In [31]:
# Nan 데이터 존재 여부
print(data[data.isnull().any(axis=1)])
# 그 데이터를 삭제
data = data.dropna()

Price      ret mkt_ret
Ticker                
Date                  
2016-01-04 NaN     NaN


In [32]:
# 2016~2025 삼복 날짜 (초복, 중복, 말복) 양력 기준
sambok = {
    2016: ('2016-07-17', '2016-07-27', '2016-08-16'),
    2017: ('2017-07-12', '2017-07-22', '2017-08-11'),
    2018: ('2018-07-17', '2018-07-27', '2018-08-16'),
    2019: ('2019-07-12', '2019-07-22', '2019-08-11'),
    2020: ('2020-07-16', '2020-07-26', '2020-08-15'),
    2021: ('2021-07-11', '2021-07-21', '2021-08-10'),
    2022: ('2022-07-16', '2022-07-26', '2022-08-15'),
    2023: ('2023-07-11', '2023-07-21', '2023-08-10'),
    2024: ('2024-07-15', '2024-07-25', '2024-08-14'),
    2025: ('2025-07-20', '2025-07-30', '2025-08-09')
}

# 날짜 행의 type을 datetime으로 전환
for year, dates in sambok.items():
    start = pd.to_datetime(sambok[year][0])
    middle = pd.to_datetime(sambok[year][1])
    end = pd.to_datetime(sambok[year][2])
    sambok[year] = (start, middle, end)